# Team T085 · Section E · Project 26
Phase 1: Machine Learning

Role 3 supplies fitted-model diagnostics and row-aligned scores. The recorded experiment reuses a previously inspected test partition; it is not a fresh independent holdout. Final evaluation and robustness are separate role handoffs.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'src').exists(), 'Start Jupyter in the project folder.'
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
TABLES=ROOT/'results/tables'
FIGURES=ROOT/'results/figures'
def table(name): return pd.read_csv(TABLES/name)
def figure(name): display(Image(filename=str(FIGURES/name)))


## Fixed classical comparators
All models receive the same normal-training observations and transformed features. Isolation Forest: 200 trees, 256 sampled records/tree. PCA: 15 components with full SVD and reconstruction MSE. Approximate OC-SVM: 128 Nyström RBF features (gamma=1/29), scaling, SGDOneClassSVM(nu=.01). Baselines remain fixed; exact kernel OC-SVM and LOF are outside this implementation.

In [2]:
# Score diagnostics do not depend on the later evaluation tables.
rows=[]
for name in ['isolation_forest','pca','approx_ocsvm']:
    saved=np.load(ROOT/f'results/scores/{name}_validation.npz')
    s=saved['score']
    rows.append(dict(model=name,rows=len(s),minimum=float(s.min()),
        median=float(np.median(s)),maximum=float(s.max()),finite=bool(np.isfinite(s).all())))
display(pd.DataFrame(rows))


,model,rows,minimum,median,maximum,finite
0,isolation_forest,56958,0.354269,0.414055,0.747427,True
1,pca,56958,0.045975,0.433923,225.852158,True
2,approx_ocsvm,56958,-0.001086,-0.000027,0.000689,True


In [3]:
manifest=json.loads((ROOT/'data/processed/split_manifest.json').read_text())
for name in ['gmm','isolation_forest','pca','approx_ocsvm']:
    for part in ['validation','test']:
        saved=np.load(ROOT/f'results/scores/{name}_{part}.npz')
        with np.load(ROOT/'data/processed/split_indices.npz') as ids:
            np.testing.assert_array_equal(saved['source_row'],ids[part])
        assert np.isfinite(saved['score']).all()
print('All eight model/split score arrays match the frozen row membership.')

All eight model/split score arrays match the frozen row membership.


## Evaluation handoff
Role 4 evaluates these fixed configurations using the same row IDs, anomaly-score direction and validation-calibrated policies. Model-family conclusions require evaluation evidence. Do not retune these baselines after test inspection.

In [4]:
# Optional comparator reproduction; this cell does not import Role 4 metrics.
RETRAIN_BASELINES=False
if RETRAIN_BASELINES:
    import yaml,joblib
    from src.data import load_raw,load_split_indices
    from src.features import prepare
    from src.baselines import fit_baselines,score as baseline_score
    from threadpoolctl import threadpool_limits
    cfg=yaml.safe_load((ROOT/'configs/config.yaml').read_text())
    path=ROOT/cfg['data_path']
    parts=load_split_indices(load_raw(path),ROOT/'data/processed/split_indices.npz',path,cfg['split_fractions'])
    xtrain,x,_=prepare(parts)
    with threadpool_limits(limits=2):
        models=fit_baselines(xtrain,cfg['seed'])
        for name,model in models.items():
            joblib.dump(model,ROOT/f'results/models/{name}.joblib')
            for split in ['validation','test']:
                ss=baseline_score(name,model,x[split])
                assert np.isfinite(ss).all()
                np.savez_compressed(ROOT/f'results/scores/{name}_{split}.npz',
                    score=ss,source_row=parts[split].index.to_numpy())
    print('Regenerated local comparators and six score arrays.')
